# Latent-Play: A Playable World Model Learned from Action-Free Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/latent-play/blob/refactor/dynamics-lam-overhaul/notebooks/colab_training_and_demo.ipynb)

This notebook executes the end-to-end pipeline of **`latent-play`** on Google Colab Pro (**A100 GPU + High-RAM**) with **automatic checkpoint persistence to Google Drive every 5 minutes**:
- If Colab disconnects at any point, clicking **Run all** will automatically resume training from the latest checkpoint saved in your Google Drive (`MyDrive/latent_play_overhaul/`).
- Datasets (`data/`) are stored on Colab's local NVMe disk for maximum read throughput (`np.memmap`), while `checkpoints/` and `outputs/` are symlinked directly to your Google Drive.

## 1. Setup, Google Drive Mounting, and Repository Cloning

In [1]:
import os, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

CUDA Available: True
Device: NVIDIA A100-SXM4-40GB
VRAM: 42.41 GB
BF16 Supported: True


In [2]:
# 1. Cleanly clone repository strictly on refactor/dynamics-lam-overhaul
import os
BRANCH = "refactor/dynamics-lam-overhaul"
!rm -rf /content/latent-play
!git clone -b {BRANCH} https://github.com/rivas-vega/latent-play.git /content/latent-play
%cd /content/latent-play
!git branch -v
os.environ["PYTHONPATH"] = "/content/latent-play"

# 2. Mount Google Drive and symlink to a clean isolated directory
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/latent_play_overhaul"
    os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/outputs", exist_ok=True)
    for folder in ("checkpoints", "outputs"):
        if os.path.islink(folder):
            os.unlink(folder)
        elif os.path.isdir(folder):
            !rm -rf {folder}
        os.symlink(f"{DRIVE_ROOT}/{folder}", folder)
    print(f"Checkpoints and outputs symlinked to Google Drive: {DRIVE_ROOT}")
except Exception as e:
    print(f"Running without Google Drive ({e}): saving to local disk.")

# 3. Install dependencies and run the 47 unit tests
!pip install -q -e ".[crafter,dev]" matplotlib
!pytest -q


Cloning into '/content/latent-play'...
remote: Enumerating objects: 552, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 552 (delta 41), reused 53 (delta 29), pack-reused 464 (from 1)
Receiving objects: 100% (552/552), 27.48 MiB | 6.04 MiB/s, done.
Resolving deltas: 100% (283/283), done.
/content/latent-play
* refactor/dynamics-lam-overhaul 3f4dbe5 chore(colab): streamline notebook exclusively for Crafter (remove ToyGrid cells)
Mounted at /content/drive
Checkpoints and outputs symlinked to Google Drive: /content/drive/MyDrive/latent_play_overhaul
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.6

## 2. Crafter Experiment: Quick Validation vs. Full Benchmark
Crafter is the primary open-world survival benchmark for `latent-play`.
Select `fast_validation` for a ~12 min end-to-end smoke test, or `full_benchmark` for the final 30K-step paper run.

In [3]:
# @title Select Crafter Experiment Scale
# @markdown Choose between a fast 12-minute smoke test or the full 30k-step benchmark.
CRAFTER_MODE = "fast_validation"  # @param ["fast_validation", "full_benchmark"]

if CRAFTER_MODE == "fast_validation":
    CRAFTER_STEPS = 50000
    TRAIN_STEPS = 2500
    WARMUP_STEPS = 250
    EVAL_EVERY = 500
    SWEEP_EPISODES = 10
    SWEEP_SEEDS = "0"
    SWEEP_FRACTIONS = "0.01 0.05 0.1 1.0"
else:
    CRAFTER_STEPS = 300000
    TRAIN_STEPS = 30000
    WARMUP_STEPS = 1000
    EVAL_EVERY = 2000
    SWEEP_EPISODES = 30
    SWEEP_SEEDS = "0 1 2"
    SWEEP_FRACTIONS = "0.001 0.01 0.05 0.1 1.0"

print(f"Active Crafter Mode: {CRAFTER_MODE}")
print(f"Data steps: {CRAFTER_STEPS} | Model train steps: {TRAIN_STEPS} | Eval episodes: {SWEEP_EPISODES}")


Active Crafter Mode: fast_validation
Data steps: 50000 | Model train steps: 2500 | Eval episodes: 10


In [4]:
!if [ ! -f data/crafter/meta.json ]; then python scripts/collect_data.py --config configs/collect_crafter.yaml --total-steps {CRAFTER_STEPS} --workers 4; fi
!python scripts/inspect_data.py --data data/crafter --out outputs/crafter_samples.gif


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide a

## 3. Train Latent Action Model (LAM) on `Crafter` (`label_fraction=0.0`)
Automatically saves checkpoints every 5 minutes to your Google Drive (`MyDrive/latent_play_overhaul/checkpoints/lam_crafter`).

In [5]:
!python scripts/train_lam.py --config configs/lam_crafter.yaml --force train.compile=true train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}


device=cuda  train windows=46628  val windows=2812
LAM params: 31.39M
--force specified: discarding checkpoints in checkpoints/lam_crafter and training from step 0.
W1019 17:36:50.877000 3706 torch/_dynamo/variables/tensor.py:1379] [3/0] Graph break from `Tensor.item()`, consider setting:
W1019 17:36:50.877000 3706 torch/_dynamo/variables/tensor.py:1379] [3/0]     torch._dynamo.config.capture_scalar_outputs = True
W1019 17:36:50.877000 3706 torch/_dynamo/variables/tensor.py:1379] [3/0] or:
W1019 17:36:50.877000 3706 torch/_dynamo/variables/tensor.py:1379] [3/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W1019 17:36:50.877000 3706 torch/_dynamo/variables/tensor.py:1379] [3/0] to include these operations in the captured graph.
W1019 17:36:50.877000 3706 torch/_dynamo/variables/tensor.py:1379] [3/0] 
W1019 17:36:50.877000 3706 torch/_dynamo/variables/tensor.py:1379] [3/0] Graph break: from user code at:
W1019 17:36:50.877000 3706 torch/_dynamo/variables/tensor.py:1379] [3/0]   File "/co

## 4. Train JEPA Dynamics + SIGReg on `Crafter` (Latent Actions vs. Oracle)
Automatically saves checkpoints every 5 minutes to your Google Drive (`dyn_crafter_lam` and `dyn_crafter_oracle`).

In [6]:
# 5a. Our model (latent-play): conditioned purely on discovered LAM latent codes
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml --force lam_checkpoint=checkpoints/lam_crafter/lam_final.pt train.out_dir=checkpoints/dyn_crafter_lam train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}

# 5b. Action Oracle (Upper Bound): conditioned on 100% ground-truth Crafter actions
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml --force train.out_dir=checkpoints/dyn_crafter_oracle train.amp=auto train.steps={TRAIN_STEPS} train.warmup={WARMUP_STEPS} train.eval_every={EVAL_EVERY}


device=cuda  train windows=43186  val windows=2617
loading pretrained LAM from checkpoints/lam_crafter/lam_final.pt...
Dynamics conditioned on 8 latent actions!
--force specified: discarding checkpoints in checkpoints/dyn_crafter_lam and training from step 0.
time=37.2  step=100  loss=0.1587  pred_loss=0.08792  var_loss=0  cov_loss=0  sigreg_loss=0.2285  eff_rank=86.2  tau=0.996  gnorm=6.173  lr=0.00012  it_s=2.697  roll_loss=0.1186  dec_loss=0.04179
time=72.9  step=200  loss=0.168  pred_loss=0.1036  var_loss=0  cov_loss=0  sigreg_loss=0.2139  eff_rank=79.96  tau=0.9961  gnorm=4.071  lr=0.00024  it_s=2.807  roll_loss=0.1073  dec_loss=0.02557
time=108.6  step=300  loss=0.1572  pred_loss=0.09165  var_loss=0  cov_loss=0  sigreg_loss=0.1866  eff_rank=77.63  tau=0.9961  gnorm=3.132  lr=0.0002997  it_s=2.809  roll_loss=0.1124  dec_loss=0.01928
time=144.3  step=400  loss=0.127  pred_loss=0.0769  var_loss=0  cov_loss=0  sigreg_loss=0.2271  eff_rank=76.73  tau=0.9962  gnorm=3.117  lr=0.0002969 

## 5. Label Fraction Sweep (0.1% to 100%), CEM Planning, and VoE Physical Scorer


In [ ]:
!python scripts/sweep_labels.py \
    --use-state-context \
    --data-root data/crafter \
    --lam-ckpt checkpoints/lam_crafter/lam_final.pt \
    --dyn-ckpt checkpoints/dyn_crafter_lam/dyn_final.pt \
    --oracle-ckpt checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --fractions {SWEEP_FRACTIONS} \
    --num-episodes {SWEEP_EPISODES} \
    --seeds {SWEEP_SEEDS} \
    --out-dir outputs/sweep_results

!python scripts/eval_voe.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --lam-checkpoint checkpoints/lam_crafter/lam_final.pt \
    --out outputs/voe_metrics.json

!python scripts/plot_paper_figures.py --metrics-json outputs/sweep_results/sweep_metrics.json --out-dir outputs/figures


Running label-fraction sweep on fractions: [0.01, 0.05, 0.1, 1.0]
Seeds: [0] | Use State Context: True | Collapse Blocked: False
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Oracle (100% True Actions) Goal Planning Success: 0.0%
Pre-caching validation representations...
Pre-caching full training representations...
Fraction   1.0% | Seed 0 | Probe Acc:  16.2% | Planning Success:  10.0%
Fraction   5.0% | Seed 0 | Probe Acc:  16.8% | Planning Success:  10.0%
Fraction  10.0% | Seed 0 | Probe Acc:  17.6% | Planning Success:  10.0%
Fraction 100.0% | Seed 0 | Probe Acc:  22.7% | Planning Success:  10.0%
saved sweep results to outputs/sweep_results/sweep_metrics.json


In [ ]:
!python scripts/eval_planning.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --align-ckpt outputs/sweep_results/alignment_f0.01_s0.pt \
    --target-dist 4 \
    --num-episodes {SWEEP_EPISODES}

from IPython.display import Image
Image("outputs/eval_planning/planned_trajectory.gif")


## 6. Package Results and Logs for Download


In [ ]:
!zip -r latent_play_results.zip outputs/ checkpoints/*/log.jsonl checkpoints/*/confusion.json checkpoints/*/config.json
from google.colab import files
files.download("latent_play_results.zip")